# 09s — Uji Kewajaran (Sanity Check) Masukan Citra — Kaggle

**Pertanyaan yang dijawab:** apakah jawaban SVLM yang konstan disebabkan model **tidak
memakai citra sama sekali** (jawaban dari teks prompt saja), atau model **melihat citra tetapi
tidak mampu menilai kerusakan** bangunan berukuran ±30 piksel?

**Rancangan (ditetapkan sebelum dijalankan):**

| Kode | Citra | Pertanyaan | Harapan bila model memakai citra |
|---|---|---|---|
| `K-asli` | chip pasca asli | prompt kelas v4 | berbeda dari K-hitam/K-derau |
| `K-hitam` | citra hitam polos 448 | prompt kelas v4 | idealnya `tidak_jelas` |
| `K-derau` | derau acak 448 | prompt kelas v4 | idealnya `tidak_jelas` |
| `F-asli` / `F-hitam` | pra+pasca asli / dua citra hitam | prompt fitur v4 | fitur berbeda antara asli dan hitam |
| `T-warna` | chip pasca asli | "Apa warna kotak di tengah citra?" | magenta / ungu / merah muda |
| `T-ada` | chip pasca asli | "Adakah bangunan di dalam kotak?" | YA |
| `T-ada-hitam` | citra hitam | pertanyaan yang sama | TIDAK |
| `T-atap` | chip pasca asli | "Apa warna atap bangunan sasaran?" | bervariasi antarbangunan |

Bangunan: 6 bangunan dari **set pengembangan** (2 per kelas; set yang sama dengan 09p,
dikeluarkan dari metrik akhir). Kedua model, satu sesi, **± 10 menit**.
Hasil disimpan sebagai run berjenis `kewajaran` di `runs/`.

## Sel 1 — pustaka

In [ ]:
!pip install -q "transformers==4.51.3" accelerate timm einops sentencepiece
import sys, torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "GPU tidak aktif"

## Sel 2 — data dan rancangan uji

In [ ]:
import json, importlib.util, datetime, re, time, gc
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

INPUT, KERJA = Path("/kaggle/input"), Path("/kaggle/working")
def cari_satu(pola):
    h = sorted(INPUT.rglob(pola))
    if not h: raise FileNotFoundError(pola)
    return h[-1]

KONFIG = json.loads(cari_satu("konfigurasi_fitur.json").read_text(encoding="utf-8"))
MP = json.loads(cari_satu("manifes_v*_*.json").read_text(encoding="utf-8"))
PROMPT = {j: cari_satu(MP["prompt"][j]["berkas"]).read_text(encoding="utf-8") for j in ("fitur", "kelas")}
spec = importlib.util.spec_from_file_location("parser05", cari_satu("05_parser_keluaran.py"))
P = importlib.util.module_from_spec(spec); spec.loader.exec_module(P)

besar = pd.read_csv(cari_satu("sampel_besar.csv")); kecil = pd.read_csv(cari_satu("sampel_kecil.csv"))
calon = besar[~besar["uid"].isin(kecil["uid"])]
DEV = pd.concat([g.sample(5, random_state=42) for _, g in calon.groupby("kelas_xbd")])   # = set 09p
UJI = pd.concat([g.head(2) for _, g in DEV.groupby("kelas_xbd")]).reset_index(drop=True)
peta = {f.name: f for f in INPUT.rglob("*.png") if f.parent.name in ("pra", "pasca")}
UJI["f_pra"], UJI["f_pasca"] = UJI["chip_pra"].map(peta), UJI["chip_pasca"].map(peta)
assert UJI[["f_pra", "f_pasca"]].notna().all().all()

HITAM = Image.new("RGB", (448, 448), (0, 0, 0))
DERAU = Image.fromarray(np.random.default_rng(42).integers(0, 256, (448, 448, 3), dtype=np.uint8))
PANDU = "Perhatikan kotak berwarna di tengah citra satelit ini dan bidik silang di dalamnya. "
TANYA = {
    "T-warna": PANDU + "Apa warna kotak tersebut? Jawab dengan satu kata.",
    "T-ada": PANDU + "Apakah di dalam kotak itu terdapat bangunan? Jawab hanya YA atau TIDAK.",
    "T-atap": PANDU + "Apa warna atap bangunan yang ditunjuk bidik silang? Jawab satu atau dua kata.",
}
print(UJI[["uid", "kelas_xbd"]])

## Sel 3 — fungsi model (sama dengan notebook 09p)

In [ ]:
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from transformers import (Qwen2_5_VLForConditionalGeneration, AutoProcessor,
                          AutoModel, AutoTokenizer)
REPO = {"qwen2.5-vl-3b": "Qwen/Qwen2.5-VL-3B-Instruct", "internvl3-2b": "OpenGVLab/InternVL3-2B"}
buka = lambda f: Image.open(f).convert("RGB")

def muat(nama):
    if nama.startswith("qwen"):
        m = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            REPO[nama], torch_dtype=torch.float16, device_map={"": 0}).eval()
        return {"nama": nama, "m": m, "proc": AutoProcessor.from_pretrained(REPO[nama])}
    try:
        m = AutoModel.from_pretrained(REPO[nama], torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
                                      use_flash_attn=False, trust_remote_code=True).eval().cuda()
    except AttributeError:
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(
            lambda self: list(getattr(self, "_tied_weights_keys", None) or []))
        m = AutoModel.from_pretrained(REPO[nama], torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
                                      use_flash_attn=False, trust_remote_code=True).eval().cuda()
    return {"nama": nama, "m": m,
            "tok": AutoTokenizer.from_pretrained(REPO[nama], trust_remote_code=True, use_fast=False)}

TF = T.Compose([T.Resize((448, 448), interpolation=InterpolationMode.BICUBIC), T.ToTensor(),
                T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])

def tanya(M, gambar, teks, maks=96):
    if M["nama"].startswith("qwen"):
        proc, m = M["proc"], M["m"]
        isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
        chat = proc.apply_chat_template([{"role": "user", "content": isi}], tokenize=False,
                                        add_generation_prompt=True)
        x = proc(text=[chat], images=gambar, return_tensors="pt", padding=True).to(m.device)
        with torch.inference_mode():
            y = m.generate(**x, max_new_tokens=maks, do_sample=False)
        return proc.decode(y[0][x["input_ids"].shape[1]:], skip_special_tokens=True)
    px = [TF(g).unsqueeze(0).to(torch.bfloat16).cuda() for g in gambar]
    awal = "<image>\n" if len(px) == 1 else "".join(f"Citra-{i}: <image>\n" for i in range(1, len(px) + 1))
    return M["m"].chat(M["tok"], torch.cat(px, 0), awal + teks,
                       dict(max_new_tokens=maks, do_sample=False), num_patches_list=[1] * len(px))

def skor_opsi_qwen(M, gambar, teks, opsi):
    # log-peluang total token jawaban untuk setiap pilihan, diberi prompt dan citra yang sama
    proc, m = M["proc"], M["m"]
    isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
    awal = proc.apply_chat_template([{"role": "user", "content": isi}], tokenize=False,
                                    add_generation_prompt=True)
    id_awal = proc(text=[awal], images=gambar, return_tensors="pt")["input_ids"][0]
    n_awal = len(id_awal)
    hasil = {}
    for o in opsi:
        x = proc(text=[awal + o], images=gambar, return_tensors="pt").to(m.device)
        ids = x["input_ids"][0]
        # awalan harus tertokenisasi identik; bila tidak, skor pilihan ini tidak sah
        if not torch.equal(ids[:n_awal].cpu(), id_awal):
            hasil[o] = float("nan"); continue
        with torch.inference_mode():
            lp = torch.log_softmax(m(**x).logits[0].float(), -1)
        sasaran = ids[n_awal:]
        hasil[o] = lp[n_awal - 1:-1].gather(1, sasaran.unsqueeze(1)).sum().item()
    return hasil

def lepas(M):
    del M["m"]; gc.collect(); torch.cuda.empty_cache()

## Sel 4 — jalankan

In [ ]:
MULAI = datetime.datetime.now(datetime.timezone.utc)
RUN_ID = f"{MULAI:%Y%m%d-%H%M%S}_kewajaran_v{MP['versi_prompt']}_{MP['sidik_fitur']}"
DIR = KERJA / "runs" / RUN_ID; DIR.mkdir(parents=True)
BARIS = []
for nama in ("qwen2.5-vl-3b", "internvl3-2b"):
    M = muat(nama); print(nama, "dimuat")
    for r in UJI.itertuples(index=False):
        pra, pasca = buka(r.f_pra), buka(r.f_pasca)
        tugas = [("K-asli", [pasca], PROMPT["kelas"], 64), ("K-hitam", [HITAM], PROMPT["kelas"], 64),
                 ("K-derau", [DERAU], PROMPT["kelas"], 64),
                 ("F-asli", [pra, pasca], PROMPT["fitur"], 256), ("F-hitam", [HITAM, HITAM], PROMPT["fitur"], 256),
                 ("T-warna", [pasca], TANYA["T-warna"], 12), ("T-ada", [pasca], TANYA["T-ada"], 8),
                 ("T-ada-hitam", [HITAM], TANYA["T-ada"], 8), ("T-atap", [pasca], TANYA["T-atap"], 12)]
        for kode, gbr, teks, maks in tugas:
            t0 = time.time()
            jwb = tanya(M, gbr, teks, maks)
            BARIS.append({"model": nama, "uid": r.uid, "kelas_xbd": r.kelas_xbd, "uji": kode,
                          "mentah": jwb, "detik": round(time.time() - t0, 2)})
            print(f"  {nama[:6]} uid={r.uid[:8]} {r.kelas_xbd:<13} {kode:<12} -> {jwb.strip()[:70]!r}")
    lepas(M)
with (DIR / "mentah.jsonl").open("w", encoding="utf-8") as fh:
    for b in BARIS: fh.write(json.dumps(b, ensure_ascii=False) + "\n")
(DIR / "manifest.json").write_text(json.dumps({
    "run_id": RUN_ID, "jenis": "kewajaran", "status": "selesai",
    "waktu_mulai_utc": MULAI.isoformat(timespec="seconds"),
    "tujuan": "memisahkan 'model tidak memakai citra' dari 'model melihat tetapi tidak menilai kerusakan'",
    "uid": UJI["uid"].tolist(), "rancangan": {k: v for k, v in TANYA.items()},
    "lingkungan": {"torch": torch.__version__, "transformers": transformers.__version__,
                   "gpu": torch.cuda.get_device_name(0)},
}, ensure_ascii=False, indent=2), encoding="utf-8")

## Sel 5 — baca hasil

- **Kelas**: bila `K-asli`, `K-hitam`, dan `K-derau` memberi kelas yang sama untuk hampir semua
  bangunan, jawaban kelas berasal dari teks prompt, bukan citra.
- **Pertanyaan mudah**: bila `T-warna` benar dan `T-ada` ≠ `T-ada-hitam`, model *memakai* citra;
  kegagalan kelas berarti ketidakmampuan menilai kerusakan, bukan kebutaan input.

In [ ]:
df = pd.DataFrame(BARIS)
def kls(t):
    return P.urai_kelas(t, KONFIG)["kelas_kerusakan"]
def ftr(t):
    h = P.urai_fitur(t, KONFIG)
    return "|".join(str(h.get("fitur_" + k)) for k in KONFIG["seleksi"]["terpilih"])
L = []
for m, g in df.groupby("model"):
    L.append(f"\n===== {m} =====")
    k = g[g.uji.str.startswith("K-")].copy(); k["kelas"] = k.mentah.map(kls)
    L.append("Kelas per kondisi (baris = bangunan):\n" +
             k.pivot_table(index=["kelas_xbd", "uid"], columns="uji", values="kelas", aggfunc="first").to_string())
    sama = (k.pivot_table(index="uid", columns="uji", values="kelas", aggfunc="first")
             .nunique(axis=1) == 1).mean()
    L.append(f"proporsi bangunan dengan kelas identik di asli/hitam/derau: {sama:.2f}")
    f = g[g.uji.str.startswith("F-")].copy(); f["fitur"] = f.mentah.map(ftr)
    L.append("Fitur asli vs hitam:\n" + f.pivot_table(index="uid", columns="uji", values="fitur", aggfunc="first").to_string())
    t = g[g.uji.str.startswith("T-")].copy(); t["jawab"] = t.mentah.str.strip().str[:30]
    L.append("Pertanyaan mudah:\n" + t.pivot_table(index=["kelas_xbd", "uid"], columns="uji", values="jawab", aggfunc="first").to_string())
teks = "\n".join(L); print(teks)
(DIR / "ringkasan.txt").write_text(teks, encoding="utf-8")
import shutil
print("\narsip:", shutil.make_archive(str(KERJA / f"run_{RUN_ID}"), "zip", root_dir=KERJA / "runs", base_dir=RUN_ID))